# Data Exploration — TSP + TBM Monitoring\n\nLoad raw data, inspect distributions, check alignment.

In [ ]:
import sys\nfrom pathlib import Path\nsys.path.insert(0, "..")\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport yaml\n\nsns.set_theme(style="whitegrid")\n%matplotlib inline

In [ ]:
# Load config\nwith open("../config/bsll_dyk1017_205.yaml") as f:\n    cfg = yaml.safe_load(f)\n\nfrom src.data.tsp_loader import load_tsp, build_voxel_field\nfrom src.data.monitoring import load_monitoring, aggregate_by_chainage\n\n# When data is available, uncomment:\n# tsp_df = load_tsp(cfg["data"]["tsp_path"])\n# mon_df = load_monitoring(cfg["data"]["monitoring_path"])\n# print(f"TSP voxels: {len(tsp_df)}")\n# print(f"Monitoring records: {len(mon_df)}")

## TSP Voxel Data\nExpected columns: X, Y, Z, Vp, Vs, E, Vp_Vs, Pr, rho

In [ ]:
# Uncomment when data available:\n# coords, attrs = build_voxel_field(tsp_df)\n# print(f"Coordinate range: X=[{coords[:,0].min():.1f}, {coords[:,0].max():.1f}], "\n#       f"Y=[{coords[:,1].min():.1f}, {coords[:,1].max():.1f}], "\n#       f"Z=[{coords[:,2].min():.1f}, {coords[:,2].max():.1f}]")\n# \n# fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n# attr_names = ["Vp", "Vs", "E", "Vp_Vs", "Pr", "rho"]\n# for ax, name in zip(axes.flat, attr_names):\n#     ax.hist(tsp_df[name], bins=50, color="steelblue", edgecolor="white")\n#     ax.set_title(name)\n# plt.tight_layout()

## TBM Monitoring Data\nExpected columns: chainage, AdvanceRate, RPM, Torque, Thrust, Penetration, ShieldPressure

In [ ]:
# Uncomment when data available:\n# mon_agg = aggregate_by_chainage(mon_df, step_length=cfg["excavation"]["step_length"])\n# fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n# for ax, col in zip(axes.flat, ["AdvanceRate", "RPM", "Torque", "Thrust", "Penetration", "ShieldPressure"]):\n#     ax.plot(mon_agg["chainage"], mon_agg[col], linewidth=0.5)\n#     ax.set_title(col)\n#     ax.set_xlabel("Chainage")\n# plt.tight_layout()

## TBM Parametric Geometry\nGenerate and visualize TBM surface nodes.

In [ ]:
from src.data.tbm_geometry import build_tbm_surface\n\ntbm_cfg = cfg["tbm_geometry"]\nsurface = build_tbm_surface(\n    cutterhead_radius=tbm_cfg["cutterhead_radius"],\n    shield_radius=tbm_cfg["shield_radius"],\n    front_len=tbm_cfg["front_shield_len"],\n    middle_len=tbm_cfg["middle_shield_len"],\n    tail_len=tbm_cfg["tail_shield_len"],\n    resolution=tbm_cfg["surface_resolution"],\n)\n\nfig = plt.figure(figsize=(12, 5))\nax = fig.add_subplot(121, projection="3d")\nfor comp_id, name in {0: "cutterhead", 1: "front_shield", 2: "middle_shield", 3: "tail_shield"}.items():\n    mask = surface.components == comp_id\n    ax.scatter(*surface.positions[mask].T, s=2, label=name, alpha=0.6)\nax.set_xlabel("X (advance)")\nax.set_ylabel("Y")\nax.set_zlabel("Z")\nax.legend(markerscale=5)\nax.set_title(f"TBM Surface ({len(surface)} nodes)")\n\nax2 = fig.add_subplot(122)\ncomp_counts = [np.sum(surface.components == i) for i in range(4)]\nax2.bar(["cutterhead", "front", "middle", "tail"], comp_counts, color="steelblue")\nax2.set_ylabel("Node count")\nax2.set_title("Nodes per component")\nplt.tight_layout()